# GIVP C++ - Benchmark Literature Comparison

Comparacao de **GIVP-full** vs **GRASP-only** em Sphere, Rosenbrock, Rastrigin e Ackley.
Workflow: Python no notebook orquestra compilacao/execucao de um binario C++ e calcula estatisticas.

## 1. Setup

In [ ]:
"""Notebook utilities for compiling and running the C++ literature benchmark."""

from __future__ import annotations

import json
import os
import shutil
import subprocess
from pathlib import Path

import pandas as pd
from scipy import stats

ROOT = Path.cwd().resolve().parents[1]
CPP_DIR = ROOT / "cpp"
WORK_DIR = ROOT / "Notebooks" / "_cpp_lit_compare"
WORK_DIR.mkdir(parents=True, exist_ok=True)


def find_cxx() -> str:
    """Return the first available C++ compiler executable path."""
    for c in ("g++", "clang++", "cl"):
        compiler_path = shutil.which(c)
        if compiler_path:
            return compiler_path
    raise RuntimeError("Nenhum compilador C++ encontrado.")


CXX = find_cxx()
N_RUNS = 30
DIMS = 10
MAX_ITERS = 100
print(f"Compilador: {CXX}")

## 2. Build e execucao

In [ ]:
CPP_CODE = f"""
#include <algorithm>
#include <chrono>
#include <cmath>
#include <cstdint>
#include <iostream>
#include <string>
#include <utility>
#include <vector>
#include <givp/api.hpp>
#include <givp/config.hpp>

double sphere(const std::vector<double>& x) {{
  double s = 0.0;
  for (double v : x) s += v * v;
  return s;
}}

double rosenbrock(const std::vector<double>& x) {{
  double s = 0.0;
  for (std::size_t i = 0; i + 1 < x.size(); ++i) {{
    const double a = x[i + 1] - x[i] * x[i];
    const double b = 1.0 - x[i];
    s += 100.0 * a * a + b * b;
  }}
  return s;
}}

double rastrigin(const std::vector<double>& x) {{
  const double pi = 3.14159265358979323846;
  double s = 10.0 * static_cast<double>(x.size());
  for (double v : x) s += v * v - 10.0 * std::cos(2.0 * pi * v);
  return s;
}}

double ackley(const std::vector<double>& x) {{
  const double pi = 3.14159265358979323846;
  const double n = static_cast<double>(x.size());
  double s1 = 0.0;
  double s2 = 0.0;
  for (double v : x) {{
    s1 += v * v;
    s2 += std::cos(2.0 * pi * v);
  }}
  s1 /= n;
  s2 /= n;
  return -20.0 * std::exp(-0.2 * std::sqrt(s1)) - std::exp(s2) + 20.0 + std::exp(1.0);
}}

givp::GivpConfig make_full(unsigned int seed) {{
  givp::GivpConfig cfg;
  cfg.seed = seed;
  cfg.max_iterations = {MAX_ITERS};
  cfg.alpha = 0.12;
  cfg.adaptive_alpha = true;
  cfg.alpha_min = 0.08;
  cfg.alpha_max = 0.18;
  cfg.vnd_iterations = 200;
  cfg.ils_iterations = 10;
  cfg.perturbation_strength = 4;
  cfg.use_elite_pool = true;
  cfg.elite_size = 7;
  cfg.path_relink_frequency = 8;
  cfg.use_cache = true;
  cfg.cache_size = 10000;
  cfg.early_stop_threshold = 80;
  cfg.use_convergence_monitor = true;
  return cfg;
}}

givp::GivpConfig make_grasp_only(unsigned int seed) {{
  givp::GivpConfig cfg;
  cfg.seed = seed;
  cfg.max_iterations = {MAX_ITERS};
  cfg.alpha = 0.12;
  cfg.adaptive_alpha = false;
  cfg.vnd_iterations = 1;
  cfg.ils_iterations = 1;
  cfg.perturbation_strength = 0;
  cfg.use_elite_pool = false;
  cfg.use_convergence_monitor = false;
  cfg.use_cache = true;
  cfg.cache_size = 10000;
  cfg.early_stop_threshold = {MAX_ITERS};
  return cfg;
}}

int main() {{
  struct Problem {{
    std::string name;
    double (*fn)(const std::vector<double>&);
    std::pair<double, double> bound;
  }};

  const std::vector<Problem> problems = {{
      {{"Sphere", sphere, {{-5.12, 5.12}}}},
      {{"Rosenbrock", rosenbrock, {{-5.0, 10.0}}}},
      {{"Rastrigin", rastrigin, {{-5.12, 5.12}}}},
      {{"Ackley", ackley, {{-32.768, 32.768}}}},
  }};

  for (const auto& algo : std::vector<std::string>{{"GIVP-full", "GRASP-only"}}) {{
    for (const auto& p : problems) {{
      auto bounds = std::vector<std::pair<double, double>>({DIMS}, p.bound);
      for (unsigned int seed = 0; seed < {N_RUNS}; ++seed) {{
        auto cfg = algo == "GIVP-full" ? make_full(seed) : make_grasp_only(seed);
        auto t0 = std::chrono::high_resolution_clock::now();
        auto res = givp::givp(p.fn, bounds, cfg);
        auto dt = std::chrono::duration<double>(std::chrono::high_resolution_clock::now() - t0).count();

        std::cout << algo << ',' << p.name << ',' << seed << ',' << res.fun << ','
                  << res.nfev << ',' << res.nit << ',' << dt << '\n';
      }}
    }}
  }}
}}
"""

src = WORK_DIR / "lit_compare.cpp"
exe = WORK_DIR / ("lit_compare.exe" if os.name == "nt" else "lit_compare")
src.write_text(CPP_CODE, encoding="utf-8")

include_dir = CPP_DIR / "include"
if Path(CXX).name.lower().startswith("cl"):
    cmd = [CXX, "/std:c++17", f"/I{include_dir}", str(src), f"/Fe:{exe}"]
else:
    cmd = [CXX, "-std=c++17", "-O2", "-I", str(include_dir), str(src), "-o", str(exe)]

try:
    subprocess.run(cmd, capture_output=True, text=True, check=True)  # noqa: S603
except subprocess.CalledProcessError as err:
    raise RuntimeError(err.stderr or err.stdout or str(err)) from err

try:
    run = subprocess.run([str(exe)], capture_output=True, text=True, check=True)  # noqa: S603
except subprocess.CalledProcessError as err:
    raise RuntimeError(err.stderr or err.stdout or str(err)) from err

rows = []
for line in run.stdout.strip().splitlines():
    algo, func, seed, fun, nfev, nit, time_s = line.split(",")
    rows.append(
        {
            "algorithm": algo,
            "function": func,
            "seed": int(seed),
            "fun": float(fun),
            "nfev": int(nfev),
            "nit": int(nit),
            "time_s": float(time_s),
        }
    )

df = pd.DataFrame(rows)
print(f"Registros: {len(df)}")
df.head()

## 3. Estatisticas e significancia

In [ ]:
from typing import Any

summary = (
    df.groupby(["function", "algorithm"])
    .agg(
        mean=("fun", "mean"),
        std=("fun", "std"),
        best=("fun", "min"),
        median=("fun", "median"),
        worst=("fun", "max"),
        nfev=("nfev", "mean"),
        time_s=("time_s", "mean"),
    )
    .reset_index()
)
print(summary.round(6).to_string(index=False))

print("\nWilcoxon (GIVP-full < GRASP-only):")
for fn in sorted(df["function"].unique()):
    a = df[(df["function"] == fn) & (df["algorithm"] == "GIVP-full")]["fun"].to_numpy()
    b = df[(df["function"] == fn) & (df["algorithm"] == "GRASP-only")]["fun"].to_numpy()
    test_result: Any = stats.wilcoxon(a, b, alternative="less")
    stat_value = float(test_result.statistic)
    p_value = float(test_result.pvalue)
    print(
        f"- {fn:<12} W={stat_value:6.1f} p={p_value:.4f} sig={'SIM' if p_value < 0.05 else 'NAO'}"
    )

## 4. Exportar JSON

In [ ]:
out_path = Path("benchmark_literature_comparison_cpp_results.json")
payload = {
    "metadata": {
        "n_runs": N_RUNS,
        "dims": DIMS,
        "algorithms": ["GIVP-full", "GRASP-only"],
        "functions": sorted(df["function"].unique().tolist()),
    },
    "summary": json.loads(summary.to_json(orient="records")),
    "records": rows,
}
out_path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
print(f"Arquivo salvo: {out_path.resolve()}")